# Flu Vaccine Data Cleaning

### Load Libraries 

In [1]:
rm(list=ls())

require(data.table)
require(ggplot2)
require(ggpubr)
require(dplyr)


subject_color_map = c(
  "FH1001" = "#1f77b4", "FH1002" = "#ff7f0e", "FH1003" = "#279e68",
  "FH1004" = "#d62728", "FH1005" = "#aa40fc", "FH1006" = "#8c564b",
  "FH1007" = "#e377c2", "FH1008" = "#b5bd61", "FH1009" = "#17becf",
  "FH1010" = "#aec7e8", "FH1011" = "#ffbb78", "FH1012" = "#98df8a",
  "FH1014" = "#ff9896", "FH1016" = "#c5b0d5", "FH1017" = "#c49c94",
  "FH1018" = "#f7b6d2", "FH1021" = "#dbdb8d"
)

cohort_color_map = c(
  "NDMM" = "#c48cb0",
    "Healthy" = "#00470D"
)

Loading required package: data.table

Loading required package: ggplot2

Loading required package: ggpubr

Loading required package: dplyr


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
### Load in Flu Data 
fluDat = fread('../data/msd/FH1_FH2_BR2_Flu_MSD_Panel_v2.xlsx - All_IgG_Assay.csv')
meta = as.data.table(readxl::read_excel('../data/msd/ndmm-rrmm-metadata-2025.xlsx', sheet=1))

### remove technical replicates 
fluDat = fluDat[-grep('_a', fluDat$Subject)]
head(fluDat,2)

Sample,Type,Subject,Sample Kit Barcode,Cohort,Patient visit details,Assay,DaysSinceFirstVisit,Flu_Season,Signal,Adjusted Signal,Mean,Adj. Sig. Mean,CV,Calc. Concentration,Calc. Conc. Mean,Calc. Conc. CV
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<int>,<int>,<int>,<dbl>,<dbl>,<chr>,<chr>,<dbl>
PL00025-02,Plasma,BR1011,KT00025,BR1,Flu Year 1 Day 0,Flu A/Victoria (H1N1),NA,2019-2020,41031,41031,43731,43731.33,11.660626,13723.36414,14690.85478,12.41041
PL00025-02,Plasma,BR1011,KT00025,BR1,Flu Year 1 Day 0,Flu A/Hong Kong H3,NA,2019-2020,8855,8855,8819,8819.00,4.538113,4497.236335,4478.097616,4.76393


## Align with Samples Used in DIHA study

In [3]:
### Read Flu Vaxx files from DIHA study
diha_flu_msd <- fread('../data/msd/MSD_All.csv')

In [4]:
### Split FH and BRI cohorts 
fh_fluDat = fluDat[grep('FH', fluDat$Subject),]
br_fluDat = fluDat[grep('BR', fluDat$Subject),]

fh_fluDat$sample_assay=paste(fh_fluDat$Sample,fh_fluDat$Assay)
br_fluDat$sample_assay=paste(br_fluDat$Sample,br_fluDat$Assay)
diha_flu_msd$sample_assay=paste(diha_flu_msd$Sample,diha_flu_msd$Assay)
br_fluDat<-br_fluDat %>% filter(sample_assay %in% diha_flu_msd$sample_assay)

### combine back again into a single file
fluDat = rbind(fh_fluDat, br_fluDat)
fluDat$sample_assay=paste(fluDat$Sample,fluDat$Assay)

## Extract Only Match Controls from Proteomics

In [5]:
plasma <- readRDS('../manuscript-figures/inputs/olink/MM_Plasma_Olink_Final.rds')

### Extract matched donors from the proteomics data object
healthy = plasma[plasma$cohort.cohortGuid %in% c('BR1','BR2')]

### then extract only Flu Year 1 day 0 samples 
healthy = healthy[sample.visitDetails =='Flu_Y1D0']
matched_donors <- unique(healthy$subject.subjectGuid)

In [6]:
### Label Cohorts 
fluDat$Cohort <- ''
fluDat[grep('FH1', fluDat$Subject)]$Cohort <- 'FH1'
fluDat[grep('FH2', fluDat$Subject)]$Cohort <- 'FH2'
fluDat[grep('BR1', fluDat$Subject)]$Cohort <- 'Healthy'
fluDat[grep('BR2', fluDat$Subject)]$Cohort <- 'Healthy'

### ReLabel Flu Vaccine Season
fluDat$Flu_Season = gsub('flu season', '', fluDat$Flu_Season)

### Separate vaccination series 
### data by separate years of vaccines 
fluDat$Year = 'Year 1'
fluDat$Year[grep('Year 2', fluDat$`Patient visit details`)] <- "Year 2"

### Create a 'value' variable as short name for
### MSD Calculated Concentration Mean column
fluDat$Value = as.numeric(fluDat$`Calc. Conc. Mean`)

### Create new variable Var 
fluDat$Visit <- fluDat$`Patient visit details`
fluDat$Visit <- gsub('Flu Year 1','Y1', fluDat$Visit)
fluDat$Visit <- gsub('Flu Year 2','Y2', fluDat$Visit)
fluDat$Visit <- gsub('Stand-Alone','SA', fluDat$Visit)

### Establish factor levels to 
### reflect flu series 

fluDat$Visit = factor(fluDat$Visit,
                      levels = c('Y1 SA', 'Y1 Day 0','Y1 Day 7','Y1 Day 90',
                                 'Y2 SA', 'Y2 Day 0','Y2 Day 7','Y2 Day 90'))

### Retain columns of 
### interest to capture flu season
### and metadata 
fluDat = fluDat[, c('Subject','Value','Visit', 'Assay',"Year",
                    'Flu_Season','DaysSinceFirstVisit','Cohort',
                    'Patient visit details','sample_assay')]

In [7]:
### Split cohort to identify matched
### donors from proteomics 
fluDat_FH = fluDat[Cohort=='FH1']
fluDat_HC = fluDat[Cohort!='FH1']
fluDat_HC =fluDat_HC[Subject %in% matched_donors]
fluDat = rbind(fluDat_FH, fluDat_HC)
fluDat <- distinct(fluDat)

### Subject to Phuket Only
phuket = fluDat[Assay=='Flu B/Phuket HA']
phuket = unique(phuket) 

In [8]:
options(repr.plot.width = 6, repr.plot.height = 6)

### Calculate coefficient of variation for each subject's 
### vaccination series for the FH1 Subjects only
CV = phuket[Cohort =='FH1', list(FluCV = sd(Value)/mean(Value),
                                 Flu_Season = Flu_Season), by=list(Subject,Year)]

#### Keeps Subject to FH1 Cohort who were on VRD Therapy 
FH1 = CV[!Subject %in% c('FH1022','FH1024','FH1027')]
FH1 = FH1[Flu_Season != '#N/A']
FH1 = unique(FH1)

### Combine CV across both vaccination series
FH1[, FluCV_Combined:= mean(FluCV), by=Subject]
FH1[, Responder := ifelse(FluCV_Combined >0.5, 'Responder','Non-Responder')]

fluDat <- dplyr::left_join(fluDat, distinct(FH1[,c('Subject','Responder')]), by='Subject')

write.csv(FH1, file='../data/msd/output/flu_response_cv_classification.csv')
write.csv(fluDat, file='../data/msd/output/processed_flu_data.csv')